<div align="center">

**UADE — Facultad de Ciencias Económicas**

**Economía del Riesgo y de la Información (1.4.010) — 2.º cuatrimestre 2026**

# Unidad IV — Aplicaciones a las Finanzas

*Notas de Clase 4 · Decisiones de portafolio, Markowitz, CAPM y mercados de futuros*

---

Bibliografía: Markowitz (1952) · Sharpe (1964)

</div>

## Qué vas a encontrar acá

En la **Unidad III** vimos cómo un conjunto de agentes puede repartirse un riesgo dado. El teorema de Borch mostró que un reparto Pareto-óptimo iguala las utilidades marginales ponderadas entre agentes y que, por el principio de mutualidad, cada participante termina expuesto **solo al riesgo agregado** de la economía.

Un mercado financiero es, precisamente, el mecanismo que instrumenta ese reparto. Una acción o un bono son un **paquete de derechos contingentes**: prometen un flujo que depende del estado que finalmente ocurra. Comprar una cartera es elegir un perfil de ingreso contingente; diversificar es recomponerlo para transferir riesgo hacia quien puede absorberlo a menor costo.

Esta clase recorre:

1. [Rendimiento y riesgo de una cartera](#scrollTo=sec2) — por qué la covarianza, y no la varianza, gobierna el riesgo de una cartera grande.
2. [El modelo de Markowitz](#scrollTo=sec3) — la frontera eficiente y el papel de la correlación.
3. [El activo libre de riesgo](#scrollTo=sec4) — separación de Tobin y Línea del Mercado de Capitales.
4. [El CAPM de Sharpe](#scrollTo=sec5) — equilibrio, beta y la distinción entre riesgo sistemático y no sistemático.
5. [Mercados de futuros](#scrollTo=sec6) — cobertura, especulación y ratio de cobertura óptimo.
6. [El puente con los derechos contingentes](#scrollTo=sec7) — el factor de descuento estocástico.
7. [Un ejemplo con datos](#scrollTo=sec8) y los [nueve ejercicios de la guía](#scrollTo=sec9) resueltos.

> 💡 **Hilo conductor.** Si los inversores tuvieran acceso a un mercado *completo* de derechos contingentes, podrían replicar cualquier perfil de ingreso y el análisis se reduciría al de la Unidad III. En la práctica los mercados son incompletos y las distribuciones continuas. El aporte de Markowitz y Sharpe es mostrar que, si el inversor solo mira **media y varianza**, todo el problema se ordena en un plano $(\sigma,\mu)$ y admite una teoría de equilibrio con implicancias contrastables.

### Cómo usar este notebook

Ejecutá la celda de **Setup** de abajo (una sola vez) y después andá corriendo las celdas en orden. Cada ejercicio termina con un `assert` que compara el resultado numérico con la solución analítica de la guía: si alguno falla, algo se rompió.

Para volver al estado original en cualquier momento: `Entorno de ejecución → Reiniciar y ejecutar todo`.

In [ ]:
# @title Setup — ejecutá esta celda primero  { display-mode: "form" }
# Descarga el módulo compartido de la materia y el dataset de la Unidad IV.
!wget -q -O eri_utils.py https://raw.githubusercontent.com/santiagoriverti/UADE_ERI/main/src/eri_utils.py
!wget -q -O retornos_ejemplo.csv https://raw.githubusercontent.com/santiagoriverti/UADE_ERI/main/data/retornos_ejemplo.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import eri_utils as eri

eri.estilo_uade()          # paleta navy/oro de las notas de clase
SEMILLA = 1410
rng = np.random.default_rng(SEMILLA)

print("Listo. eri_utils", eri.__version__)

<a name="sec2"></a>
## 1. Rendimiento y riesgo de una cartera

Consideremos $N$ activos riesgosos. Sea $r_i$ el rendimiento por peso invertido en el activo $i$, con

$$\mu_i = E[r_i], \qquad \sigma_i^2 = \mathrm{Var}(r_i), \qquad \sigma_{ij} = \mathrm{Cov}(r_i, r_j) = \rho_{ij}\,\sigma_i\sigma_j$$

Una cartera es un vector de ponderaciones $\mathbf{w}$ que satisface la restricción presupuestaria $\mathbf{1}^\top\mathbf{w} = 1$. Usando la linealidad de la esperanza y la bilinealidad de la covarianza:

$$\mu_p = \mathbf{w}^\top\boldsymbol{\mu}, \qquad \sigma_p^2 = \mathbf{w}^\top\boldsymbol{\Sigma}\,\mathbf{w}$$

> 💡 **Idea clave.** La media de la cartera es un promedio ponderado **simple** de las medias individuales; la varianza **no lo es**. En $\sigma_p^2$ aparecen los $N$ términos de varianza propia ($w_i^2\sigma_i^2$) y los $N(N-1)$ términos de covarianza ($w_iw_j\sigma_{ij}$). El riesgo de una cartera bien diversificada está gobernado por las **covarianzas**, no por las varianzas individuales. Ahí nace la diversificación.

In [ ]:
# Dos activos con correlación moderada.
mu = np.array([0.10, 0.18])                       # rendimientos esperados anuales
Sigma = np.array([[0.15**2,            0.2*0.15*0.30],
                  [0.2*0.15*0.30,      0.30**2     ]])

w = np.array([0.6, 0.4])
mu_p, sigma_p = eri.momentos_cartera(w, mu, Sigma)

eri.resaltar("Rendimiento esperado de la cartera", mu_p)
eri.resaltar("Desvío de la cartera", sigma_p)
eri.resaltar("Promedio ponderado de los desvíos", float(w @ np.sqrt(np.diag(Sigma))))

El desvío de la cartera (16,4 %) es **menor** que el promedio ponderado de los desvíos individuales (21 %). Esa diferencia es, exactamente, la ganancia de diversificar. La media, en cambio, sí es el promedio ponderado: **diversificar reduce el riesgo sin costo en rendimiento esperado**.

### Por qué la covarianza domina en carteras grandes

Tomemos una cartera igualmente ponderada ($w_i = 1/N$), con varianza media $\bar\sigma^2$ y covarianza media $\bar c$ entre pares distintos:

$$\sigma_p^2 = \frac{1}{N}\,\bar\sigma^2 + \left(1 - \frac{1}{N}\right)\bar c \;\xrightarrow[N\to\infty]{}\; \bar c$$

El primer término —el riesgo **propio** o idiosincrático— se desvanece al diversificar. El segundo —el riesgo de **covarianza**— sobrevive.

In [ ]:
def riesgo_equiponderada(N, sigma_medio, corr_media):
    """Desvío de una cartera 1/N con varianza media y correlación media dadas."""
    var_media = sigma_medio ** 2
    cov_media = corr_media * var_media
    return np.sqrt(var_media / N + (1 - 1 / N) * cov_media)

N = np.arange(1, 101)
fig, ax = eri.figura("La diversificación tiene un piso",
                     "cantidad de activos $N$", "desvío de la cartera $\\sigma_p$")

for corr, color, etiqueta in [(0.00, eri.VERDE, "$\\bar\\rho = 0$ (activos independientes)"),
                              (0.15, eri.ORO,   "$\\bar\\rho = 0{,}15$"),
                              (0.35, eri.NAVY,  "$\\bar\\rho = 0{,}35$ (caso realista)")]:
    ax.plot(N, riesgo_equiponderada(N, 0.30, corr), color=color, label=etiqueta)
    if corr > 0:
        ax.axhline(0.30 * np.sqrt(corr), color=color, linestyle=":", linewidth=1.2)

ax.set_ylim(0, 0.32)
ax.legend()
plt.show()

eri.resaltar("Piso no diversificable con corr = 0,35", 0.30 * np.sqrt(0.35))

Si los activos fueran independientes ($\bar c = 0$), el riesgo tendería a **cero**: es exactamente el mecanismo de la Ley de Grandes Números y del *pooling* de la Unidad III. Pero los rendimientos financieros están positivamente correlacionados —todos dependen del ciclo económico—, de modo que $\bar c > 0$ y queda un **piso de riesgo no diversificable**.

Ese piso es el germen del **riesgo sistemático** que formalizará Sharpe. Nótese lo que la figura muestra: con correlación media de 0,35 y activos de 30 % de desvío, agregar más activos después de los primeros 25 o 30 casi no reduce el riesgo. Ninguna cartera de acciones, por grande que sea, baja del 17,7 %.

<a name="sec3"></a>
## 2. El modelo de Markowitz: la frontera eficiente

### El rechazo de la regla del valor presente esperado

Markowitz (1952) abre descartando la regla ingenua de **maximizar el valor presente esperado**. Su argumento es contundente: si el inversor solo maximizara el rendimiento esperado descontado, colocaría **toda** su riqueza en el activo de mayor valor esperado.

Esa regla **nunca implica diversificar**. Y como la diversificación es a la vez observada y sensata, la regla debe rechazarse tanto como hipótesis descriptiva como norma de conducta. La covarianza —ausente en la regla del valor presente— es lo que hace deseable repartir la riqueza.

**Definición (cartera eficiente).** Una cartera es *eficiente* si no existe otra factible con igual o mayor rendimiento esperado y menor varianza, ni con igual o menor varianza y mayor rendimiento. El lugar geométrico de los pares $(\sigma_p, \mu_p)$ de las carteras eficientes es la **frontera eficiente**.

### El caso de dos activos y el papel de la correlación

Con $w_1 = w$ y $w_2 = 1-w$:

$$\mu_p = w\mu_1 + (1-w)\mu_2, \qquad \sigma_p^2 = w^2\sigma_1^2 + (1-w)^2\sigma_2^2 + 2w(1-w)\rho_{12}\sigma_1\sigma_2$$

La media es lineal en $w$; la varianza es cuadrática y **decrece con $\rho_{12}$**.

In [ ]:
mu1, s1 = 0.08, 0.12          # activo 1
mu2, s2 = 0.15, 0.25          # activo 2
w_grilla = np.linspace(0, 1, 200)

fig, ax = eri.figura("Diversificación y correlación (dos activos)",
                     "$\\sigma_p$", "$\\mu_p$")

for rho, color in [(1.0, eri.NAVY), (0.0, eri.ORO), (-1.0, eri.ROJO)]:
    mu_p = w_grilla * mu1 + (1 - w_grilla) * mu2
    var_p = ((w_grilla * s1) ** 2 + ((1 - w_grilla) * s2) ** 2
             + 2 * w_grilla * (1 - w_grilla) * rho * s1 * s2)
    ax.plot(np.sqrt(var_p), mu_p, color=color, label="$\\rho = %+.0f$" % rho)

ax.scatter([s1, s2], [mu1, mu2], color="black", zorder=5, s=30)
ax.annotate("Activo 1", (s1, mu1), textcoords="offset points", xytext=(6, -14))
ax.annotate("Activo 2", (s2, mu2), textcoords="offset points", xytext=(-18, 8))
ax.set_xlim(0, 0.28)
ax.legend()
plt.show()

Los tres casos de la figura:

| $\rho_{12}$ | Forma | Interpretación |
|---|---|---|
| $+1$ | Recta: $\sigma_p = \lvert w\sigma_1 + (1-w)\sigma_2\rvert$ | No hay ganancia por diversificar; el riesgo también es un promedio ponderado |
| $-1$ | Se anula en $w = \sigma_2/(\sigma_1+\sigma_2)$ | Existe una combinación **sin riesgo**: cobertura perfecta |
| $-1 < \rho < 1$ | Hipérbola combada hacia la izquierda | Se alcanza menos riesgo que el de cualquiera de los dos activos por separado |

**Cuanto menor es la correlación, más se comba la frontera hacia la izquierda.** Ése es todo el beneficio de la diversificación, en una sola imagen.

### La frontera con $N$ activos

Con $N$ activos, la región factible es una **bala** (región convexa) cuyo borde izquierdo es la frontera de mínima varianza. Resolviendo

$$\min_{\mathbf{w}}\ \tfrac12\,\mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w} \quad \text{s.a.} \quad \mathbf{w}^\top\boldsymbol{\mu} = \mu_p,\ \ \mathbf{w}^\top\mathbf{1} = 1$$

y definiendo $A = \mathbf{1}^\top\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$, $B = \boldsymbol{\mu}^\top\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$, $C = \mathbf{1}^\top\boldsymbol{\Sigma}^{-1}\mathbf{1}$ y $D = BC - A^2 > 0$, se obtiene

$$\sigma_p^2 = \frac{C\mu_p^2 - 2A\mu_p + B}{D}$$

una **hipérbola** en el plano $(\sigma, \mu)$. Su vértice es la cartera de **mínima varianza global** (GMV):

$$\mu_{\rm GMV} = \frac{A}{C}, \qquad \sigma_{\rm GMV}^2 = \frac{1}{C}, \qquad \mathbf{w}_{\rm GMV} = \frac{\boldsymbol{\Sigma}^{-1}\mathbf{1}}{C}$$

In [ ]:
# Cuatro activos con una estructura de correlaciones realista.
activos = ["Bonos", "Value", "Growth", "Emergentes"]
mu4 = np.array([0.045, 0.095, 0.115, 0.140])
sd4 = np.array([0.060, 0.170, 0.210, 0.290])
Corr4 = np.array([[1.00, 0.15, 0.10, 0.05],
                  [0.15, 1.00, 0.72, 0.55],
                  [0.10, 0.72, 1.00, 0.60],
                  [0.05, 0.55, 0.60, 1.00]])
Sigma4 = np.outer(sd4, sd4) * Corr4

A, B, C, D = eri.frontera_abcd(mu4, Sigma4)
w_gmv, mu_gmv, sd_gmv = eri.cartera_gmv(mu4, Sigma4)

print("Escalares de la frontera:")
eri.resaltar("A", A); eri.resaltar("B", B); eri.resaltar("C", C); eri.resaltar("D", D)
print()
print("Cartera de mínima varianza global:")
display(eri.tabla({"ponderación": w_gmv}, indice=activos))
eri.resaltar("Rendimiento de la GMV", mu_gmv)
eri.resaltar("Desvío de la GMV", sd_gmv)

Notar que la GMV carga fuertemente en bonos: la varianza mínima no le interesa el rendimiento, solo el riesgo. También aparecen ponderaciones **negativas**, que representan ventas en corto. El modelo original de Markowitz las excluye ($w_i \ge 0$); acá las permitimos porque es lo que hace posible la solución cerrada. Al restringirlas, la frontera se resuelve numéricamente y queda por dentro de la analítica.

Verifiquemos la fórmula cerrada resolviendo el mismo problema con un optimizador numérico:

In [ ]:
def frontera_numerica(mu_objetivo, mu, Sigma):
    """Mínima varianza para un rendimiento objetivo, resuelto con SLSQP."""
    N = len(mu)
    restricciones = [{"type": "eq", "fun": lambda w: w.sum() - 1.0},
                     {"type": "eq", "fun": lambda w, m=mu_objetivo: w @ mu - m}]
    res = minimize(lambda w: w @ Sigma @ w, x0=np.full(N, 1.0 / N),
                   constraints=restricciones, method="SLSQP",
                   options={"ftol": 1e-14, "maxiter": 500})
    return np.sqrt(res.fun)

objetivos = np.linspace(0.05, 0.14, 10)
sd_analitica = eri.frontera_sigma(objetivos, mu4, Sigma4)
sd_numerica = np.array([frontera_numerica(m, mu4, Sigma4) for m in objetivos])

comparacion = eri.tabla({"objetivo $\\mu_p$": objetivos,
                         "analítica": sd_analitica,
                         "numérica": sd_numerica,
                         "diferencia": np.abs(sd_analitica - sd_numerica)}, decimales=10)
display(comparacion)

assert np.allclose(sd_analitica, sd_numerica, atol=1e-7), \
    "La fórmula cerrada y el optimizador deberían coincidir"
print("\nLa fórmula cerrada y el optimizador numérico coinciden hasta 1e-7.")

In [ ]:
# La bala de Markowitz: 20.000 carteras aleatorias, la frontera y la GMV.
n_carteras = 20_000
pesos = rng.dirichlet(np.ones(len(mu4)), size=n_carteras)      # solo posiciones largas
mu_nube = pesos @ mu4
sd_nube = np.sqrt(np.einsum("ij,jk,ik->i", pesos, Sigma4, pesos))

rango = np.linspace(0.02, 0.16, 300)
sd_front = eri.frontera_sigma(rango, mu4, Sigma4)
eficiente = rango >= mu_gmv

fig, ax = eri.figura("La bala de Markowitz", "$\\sigma_p$", "$\\mu_p$")
sc = ax.scatter(sd_nube, mu_nube, c=mu_nube / sd_nube, s=2, alpha=0.35, cmap="cividis")
ax.plot(sd_front[~eficiente], rango[~eficiente], color=eri.GRIS, linestyle="--",
        label="mínima varianza (ineficiente)")
ax.plot(sd_front[eficiente], rango[eficiente], color=eri.NAVY, linewidth=2.6,
        label="frontera eficiente")
ax.scatter([sd_gmv], [mu_gmv], color=eri.ORO, s=70, zorder=5, edgecolor="white",
           label="GMV")
ax.scatter(sd4, mu4, color="black", s=28, zorder=5)
for nombre, s, m in zip(activos, sd4, mu4):
    ax.annotate(nombre, (s, m), textcoords="offset points", xytext=(7, -3), fontsize=9)
ax.set_xlim(0, 0.32); ax.set_ylim(0.02, 0.16)
fig.colorbar(sc, ax=ax, label="$\\mu_p/\\sigma_p$")
ax.legend(loc="lower right")
plt.show()

Dos cosas que la figura hace evidentes y que conviene señalar en voz alta:

1. **Ninguna cartera cae a la izquierda de la frontera.** No es casualidad: la frontera es, por construcción, el mínimo de varianza para cada rendimiento. Las 20.000 carteras aleatorias son una verificación empírica del resultado analítico.
2. **La nube no toca la frontera** salvo en los extremos, porque las carteras aleatorias son solo largas mientras que la frontera analítica admite ventas en corto.

<a name="sec4"></a>
## 3. El activo libre de riesgo y el teorema de separación

Sharpe (1964), siguiendo a Tobin (1958), incorpora un **activo libre de riesgo** de rendimiento cierto $r_f$. Una cartera que coloca una fracción $a$ en un portafolio riesgoso $T$ y $1-a$ en el activo seguro tiene

$$\mu_p = r_f + a(\mu_T - r_f), \qquad \sigma_p = a\,\sigma_T$$

Eliminando $a = \sigma_p/\sigma_T$ se obtiene una **recta**:

$$\mu_p = r_f + \frac{\mu_T - r_f}{\sigma_T}\,\sigma_p$$

El inversor querrá la recta de mayor pendiente posible; esa recta es **tangente** a la frontera de activos riesgosos, y el punto de tangencia $T$ es el **portafolio tangente**:

$$\mathbf{w}_T \propto \boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu} - r_f\mathbf{1})$$

**Teorema de separación en dos fondos (Tobin).** Con un activo libre de riesgo y expectativas dadas, *todo* inversor con preferencias media-varianza elige una combinación del **mismo** portafolio riesgoso tangente $T$ y del activo seguro. La composición de la parte riesgosa es igual para todos; solo cambia la proporción $a$ según el grado de aversión al riesgo.

> 💡 **Idea clave.** La decisión se **dicotomiza**: primero se determina la mejor cartera riesgosa $T$ —un problema técnico, idéntico para todos— y después cada inversor decide cuánto arriesgar mezclando $T$ con el activo seguro —un problema de gustos. La aversión al riesgo no afecta *qué* activos riesgosos comprar, solo *cuánto* del bloque riesgoso tomar.

In [ ]:
rf = 0.03
w_tan, mu_tan, sd_tan = eri.cartera_tangente(mu4, Sigma4, rf)
sharpe = eri.ratio_sharpe(mu_tan, sd_tan, rf)

print("Portafolio tangente (= portafolio de mercado en equilibrio):")
display(eri.tabla({"ponderación": w_tan}, indice=activos))
eri.resaltar("Rendimiento del tangente", mu_tan)
eri.resaltar("Desvío del tangente", sd_tan)
eri.resaltar("Ratio de Sharpe (pendiente de la CML)", sharpe)

fig, ax = eri.figura("Frontera eficiente y Línea del Mercado de Capitales",
                     "$\\sigma_p$", "$\\mu_p$")
ax.plot(sd_front[eficiente], rango[eficiente], color=eri.NAVY, label="frontera eficiente")
ax.plot(sd_front[~eficiente], rango[~eficiente], color=eri.GRIS, linestyle="--")

sd_cml = np.linspace(0, 0.32, 50)
ax.plot(sd_cml, rf + sharpe * sd_cml, color=eri.ORO, linewidth=2.4,
        label="CML: $\\mu_p = r_f + S\\,\\sigma_p$")
ax.scatter([0], [rf], color="black", s=35, zorder=5)
ax.annotate("$r_f$", (0, rf), textcoords="offset points", xytext=(8, -4))
ax.scatter([sd_tan], [mu_tan], color=eri.ORO, s=80, zorder=6, edgecolor="white")
ax.annotate("$T = M$", (sd_tan, mu_tan), textcoords="offset points", xytext=(10, -6))
ax.scatter([sd_gmv], [mu_gmv], color=eri.GRIS, s=45, zorder=5)
ax.annotate("GMV", (sd_gmv, mu_gmv), textcoords="offset points", xytext=(6, -14), fontsize=9)
ax.set_xlim(0, 0.32); ax.set_ylim(0.02, 0.17)
ax.legend(loc="lower right")
plt.show()

La **Línea del Mercado de Capitales (CML)** describe la relación media-riesgo de las carteras **eficientes**:

$$\mu_p = r_f + \frac{\mu_M - r_f}{\sigma_M}\,\sigma_p$$

Su ordenada al origen $r_f$ es el **precio del tiempo**; su pendiente —el *ratio de Sharpe* del mercado— es el **precio del riesgo**: el rendimiento esperado adicional por cada unidad de desvío estándar soportada.

Nótese que la CML domina a la frontera en todo su recorrido salvo en el punto de tangencia. Introducir un activo seguro mejora a **todos** los inversores, sin excepción.

<a name="sec5"></a>
## 4. El CAPM de Sharpe: equilibrio y riesgo sistemático

El **Capital Asset Pricing Model** lleva el análisis de Markowitz al equilibrio de mercado. Se apoya en cuatro supuestos:

1. Inversores media-varianza y aversos al riesgo, con horizonte común de un período.
2. **Expectativas homogéneas**: todos comparten las mismas $(\mu_i, \sigma_{ij})$.
3. Existe un activo libre de riesgo a cuya tasa $r_f$ se puede prestar y pedir prestado.
4. Mercados competitivos y sin fricciones; activos perfectamente divisibles.

Bajo expectativas homogéneas todos calculan la *misma* frontera y el *mismo* portafolio tangente. Y si todos compran $T$, en el agregado $T$ debe contener a todos los activos en la proporción de su valor de mercado: **el portafolio tangente es el portafolio de mercado $M$**.

**Ecuación del CAPM.** En equilibrio, el rendimiento esperado de todo activo $i$ satisface

$$\mu_i = r_f + \beta_i\,(\mu_M - r_f), \qquad \beta_i = \frac{\mathrm{Cov}(r_i, r_M)}{\mathrm{Var}(r_M)}$$

La representación gráfica de esta ecuación es la **Línea del Mercado de Títulos (SML)**.

In [ ]:
rf_c, mu_M, sd_M = 0.04, 0.10, 0.18

betas = np.linspace(0, 2, 100)
fig, ax = eri.figura("Línea del Mercado de Títulos (SML)", "$\\beta_i$", "$\\mu_i$")
ax.plot(betas, eri.capm(betas, rf_c, mu_M), color=eri.NAVY, linewidth=2.5, label="SML")
ax.scatter([0, 1], [rf_c, mu_M], color="black", s=35, zorder=5)
ax.annotate("$r_f$", (0, rf_c), textcoords="offset points", xytext=(8, -4))
ax.annotate("$M$  ($\\beta = 1$)", (1, mu_M), textcoords="offset points", xytext=(10, -4))

# Un activo por encima de la SML y otro por debajo.
ax.scatter([0.7], [0.10], marker="^", color=eri.VERDE, s=90, zorder=6)
ax.annotate("infravalorado", (0.7, 0.10), textcoords="offset points", xytext=(-20, 10),
            color=eri.VERDE, fontsize=9)
ax.scatter([1.3], [0.09], marker="s", color=eri.ROJO, s=70, zorder=6)
ax.annotate("sobrevalorado", (1.3, 0.09), textcoords="offset points", xytext=(-10, -20),
            color=eri.ROJO, fontsize=9)
ax.set_ylim(0, 0.18)
ax.legend(loc="upper left")
plt.show()

eri.resaltar("Exigido por la SML a un activo con beta 0,70", eri.capm(0.70, rf_c, mu_M))
eri.resaltar("Alfa del activo infravalorado", 0.10 - eri.capm(0.70, rf_c, mu_M))

Un activo **por encima** de la SML ofrece más rendimiento del que corresponde a su $\beta$: está infravalorado y conviene comprarlo. Al hacerlo, su precio sube, su rendimiento esperado cae y vuelve a la recta. En equilibrio, todos los activos están **sobre** la SML.

### Riesgo sistemático y no sistemático

La clave económica del CAPM es la descomposición del riesgo. Proyectando el rendimiento del activo sobre el del mercado, $r_i = \alpha_i + \beta_i r_M + \varepsilon_i$ con $\mathrm{Cov}(r_M, \varepsilon_i) = 0$, la varianza total se parte en dos componentes **ortogonales**:

$$\underbrace{\sigma_i^2}_{\text{riesgo total}} = \underbrace{\beta_i^2\,\sigma_M^2}_{\text{sistemático}} + \underbrace{\sigma_{\varepsilon_i}^2}_{\text{no sistemático}}$$

In [ ]:
sd_i, beta_i = 0.28, 1.20
var_t, var_s, var_e, r2 = eri.descomposicion_riesgo(sd_i, beta_i, sd_M)

eri.resaltar("Varianza total", var_t)
eri.resaltar("Varianza sistemática (beta^2 * var_M)", var_s)
eri.resaltar("Varianza no sistemática (residual)", var_e)
eri.resaltar("Fracción sistemática (R cuadrado)", r2)

fig, ax = eri.figura("Descomposición del riesgo de un activo", "", "varianza")
ax.bar(["riesgo total"], [var_t], color=eri.GRIS, width=0.5)
ax.bar(["sistemático\n+ idiosincrático"], [var_s], color=eri.NAVY, width=0.5,
       label="sistemático — se paga")
ax.bar(["sistemático\n+ idiosincrático"], [var_e], bottom=[var_s], color=eri.ORO, width=0.5,
       label="idiosincrático — no se paga")
ax.grid(axis="x", visible=False)
ax.legend()
plt.show()

> 💡 **Idea clave: solo se paga el riesgo que no se puede evitar.** Como el mercado no recompensa un riesgo que el propio inversor puede anular gratis diversificando, en equilibrio el premio por riesgo de un activo depende **únicamente** de su riesgo sistemático $\beta_i$, no de su riesgo total $\sigma_i$. Ésta es la respuesta de Sharpe a la pregunta que la teoría tradicional dejaba abierta: *cuál* componente del riesgo determina el precio de un activo.

**Nota sobre la notación original de Sharpe.** El artículo de 1964 llama $P$ a la tasa pura de interés (nuestro $r_f$), escribe la CML como $\sigma_R = S(E_R - P)$, no habla todavía de «portafolio de mercado» sino de una combinación eficiente cualquiera $g$, y denota $B_{ig}$ a nuestro $\beta_i$. Un resultado central de su paper es que **todas** las combinaciones eficientes están perfectamente correlacionadas entre sí, lo que justifica tomar cualquiera —en particular el mercado— como referencia. La forma moderna se consolidó poco después con Lintner (1965) y Mossin (1966).

<a name="sec6"></a>
## 5. Mercados de futuros

Los mercados de futuros no están cubiertos por Markowitz ni por Sharpe; los desarrollamos con fuentes adicionales (Hull, 2018; Varian, 1992). Son un segundo instrumento —junto con la cartera de activos— para **transferir riesgo** entre agentes.

**Definición.** Un **contrato de futuros** es un acuerdo estandarizado, negociado en un mercado organizado, para comprar o vender un activo subyacente en una fecha futura $T$ a un precio $F_0$ pactado hoy. Quien se compromete a comprar toma una **posición larga**; quien se compromete a vender, una **posición corta**. A diferencia del *forward* (contrato a medida, extrabursátil), el futuro se ajusta a mercado diariamente y tiene cámara compensadora.

Cumple dos funciones económicas complementarias:

- **Cobertura (*hedging*).** El agente expuesto a la variación de un precio toma una posición de signo opuesto, fija de antemano el precio de su transacción futura y reduce la varianza de su resultado. Es el mismo motivo de la Unidad III: transferir riesgo hacia quien lo soporta a menor costo.
- **Especulación.** El agente sin exposición previa apuesta a la dirección del precio, aporta liquidez y absorbe el riesgo que los coberturistas quieren ceder.

**Cost-of-carry.** Para un activo de inversión almacenable, el arbitraje conduce a

$$F_0 = S_0\,e^{(r + u - y)T}$$

con $r$ la tasa libre de riesgo, $u$ el costo de almacenamiento y $y$ el rendimiento de conveniencia. El mercado está en **contango** si $F_0 > S_0$ y en ***backwardation*** si $F_0 < S_0$.

In [ ]:
S0, r_libre, T = 100.0, 0.05, 1.0
escenarios = [("Contango (activo financiero)", 0.00, 0.00),
              ("Contango fuerte (con almacenaje)", 0.04, 0.00),
              ("Backwardation (conveniencia alta)", 0.02, 0.12)]

filas = []
for nombre, u, y in escenarios:
    F0 = S0 * np.exp((r_libre + u - y) * T)
    filas.append({"escenario": nombre, "u": u, "y": y, "F_0": F0,
                  "base (F_0 - S_0)": F0 - S0,
                  "estado": "contango" if F0 > S0 else "backwardation"})

display(pd.DataFrame(filas).round(4))

### Cobertura de varianza mínima

El puente con la teoría de cartera es directo. Si un agente tiene una posición en el activo de contado y cubre con $h$ unidades de futuro, el desvío de su resultado se minimiza —un problema media-varianza— en el **ratio de cobertura óptimo**

$$h^\star = \rho_{SF}\,\frac{\sigma_S}{\sigma_F} = \frac{\mathrm{Cov}(\Delta S, \Delta F)}{\mathrm{Var}(\Delta F)}$$

que es, formalmente, el $\beta$ del precio spot respecto del precio de futuro. La varianza mínima resultante es $\sigma_S^2(1 - \rho_{SF}^2)$: la cobertura elimina una fracción $\rho_{SF}^2$ de la varianza.

In [ ]:
sd_S, sd_F, rho_SF = 0.030, 0.028, 0.90
h_opt = eri.hedge_ratio(rho_SF, sd_S, sd_F)

h_grilla = np.linspace(0, 2, 300)
var_resultado = sd_S**2 - 2 * h_grilla * rho_SF * sd_S * sd_F + (h_grilla * sd_F) ** 2

fig, ax = eri.figura("El ratio de cobertura de varianza mínima",
                     "ratio de cobertura $h$", "desvío del resultado")
ax.plot(h_grilla, np.sqrt(var_resultado), color=eri.NAVY)
ax.axvline(h_opt, color=eri.ORO, linestyle="--", label="$h^\\star = %.3f$" % h_opt)
ax.axhline(sd_S, color=eri.GRIS, linestyle=":", label="sin cobertura")
ax.scatter([h_opt], [sd_S * np.sqrt(1 - rho_SF**2)], color=eri.ORO, s=70, zorder=5)
ax.legend()
plt.show()

eri.resaltar("Ratio de cobertura óptimo h*", h_opt)
eri.resaltar("Desvío sin cobertura", sd_S)
eri.resaltar("Desvío con cobertura óptima", sd_S * np.sqrt(1 - rho_SF**2))
eri.resaltar("Reducción de la varianza (= rho^2)", rho_SF ** 2)

La curva es una parábola: **sobrecubrirse es tan malo como no cubrirse**. Con $h$ demasiado alto, la posición en futuros deja de compensar la exposición y pasa a agregar riesgo propio. Con $\rho_{SF} = 1$ se recuperaría la cobertura perfecta —el caso $\rho = -1$ de la figura de dos activos: dos posiciones perfectamente correlacionadas y de signo opuesto se cancelan.

<a name="sec7"></a>
## 6. El puente con los mercados de derechos contingentes

Cerramos volviendo al lenguaje de la Unidad III. Un activo financiero con pagos $d_i(s)$ en el estado $s$ es una combinación de **valores de Arrow**. Si hay tantos activos linealmente independientes como estados, el mercado es **completo**: cualquier perfil de ingreso contingente puede replicarse y reaparecen los precios de estado.

La conexión se ve con nitidez en la valuación. En la Unidad III el precio de un activo era $q = E[m\,x]$, con $m$ el **factor de descuento estocástico**. Reescribiendo esa identidad en términos de rendimientos se obtiene exactamente una relación de tipo *beta*:

$$\mu_i - r_f = -\,r_f\,\mathrm{Cov}(m, r_i)$$

El premio por riesgo de un activo es proporcional a la covarianza de su rendimiento con el factor de descuento. **El CAPM es el caso particular en que ese factor es lineal en el rendimiento del mercado**, $m = a - b\,r_M$: entonces la covarianza con $m$ se traduce en covarianza con el mercado y recuperamos $\mu_i = r_f + \beta_i(\mu_M - r_f)$.

> 💡 **Síntesis.** Diversificar es reordenar derechos contingentes para quedarse solo con el riesgo agregado. El $\beta$ del CAPM mide cuánto de ese riesgo agregado carga cada activo, y el mercado solo remunera esa porción. Lo idiosincrático —evitable— no se paga. Markowitz-Sharpe y Arrow-Borch son dos lecturas del mismo problema de reparto del riesgo.

<a name="sec8"></a>
## 7. Interactivo: mové los parámetros

Dos controles que muestran lo que las figuras estáticas no pueden. Si los deslizadores no aparecen, ejecutá `Entorno de ejecución → Reiniciar y ejecutar todo`.

In [ ]:
# @title Frontera de dos activos: mové la correlación  { display-mode: "form" }
def frontera_interactiva(rho=0.2):
    w_g = np.linspace(-0.2, 1.2, 300)
    mu_g = w_g * mu1 + (1 - w_g) * mu2
    var_g = ((w_g * s1) ** 2 + ((1 - w_g) * s2) ** 2
             + 2 * w_g * (1 - w_g) * rho * s1 * s2)
    sd_g = np.sqrt(var_g)

    fig, ax = eri.figura("Frontera de dos activos con $\\rho = %+.2f$" % rho,
                         "$\\sigma_p$", "$\\mu_p$")
    ax.plot(sd_g, mu_g, color=eri.NAVY)
    k = int(np.argmin(sd_g))
    ax.scatter([sd_g[k]], [mu_g[k]], color=eri.ORO, s=80, zorder=5,
               label="mínima varianza: $\\sigma = %.4f$" % sd_g[k])
    ax.scatter([s1, s2], [mu1, mu2], color="black", s=30, zorder=5)
    ax.set_xlim(0, 0.30); ax.set_ylim(0.05, 0.17)
    ax.legend(loc="upper left")
    plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(frontera_interactiva,
             rho=FloatSlider(value=0.2, min=-1.0, max=1.0, step=0.05,
                             description="correlación", continuous_update=False))
except ImportError:
    print("ipywidgets no disponible: se muestra el caso rho = 0,2.")
    frontera_interactiva(0.2)

In [ ]:
# @title Diversificación: mové N y la correlación media  { display-mode: "form" }
def diversificacion_interactiva(N_max=50, corr_media=0.30):
    Ns = np.arange(1, N_max + 1)
    sd = riesgo_equiponderada(Ns, 0.30, corr_media)
    piso = 0.30 * np.sqrt(corr_media)

    fig, ax = eri.figura("Riesgo de una cartera 1/N", "$N$", "$\\sigma_p$")
    ax.plot(Ns, sd, color=eri.NAVY, marker="o", markersize=3)
    ax.axhline(piso, color=eri.ORO, linestyle="--",
               label="piso no diversificable = %.4f" % piso)
    ax.set_ylim(0, 0.32)
    ax.legend()
    plt.show()

    eri.resaltar("Riesgo con N activos", sd[-1])
    eri.resaltar("Fracción del riesgo ya eliminada", 1 - sd[-1] / 0.30)

try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(diversificacion_interactiva,
             N_max=IntSlider(value=50, min=1, max=200, step=1, description="N",
                             continuous_update=False),
             corr_media=FloatSlider(value=0.30, min=0.0, max=0.9, step=0.05,
                                    description="corr. media", continuous_update=False))
except ImportError:
    print("ipywidgets no disponible: se muestra el caso N = 50 y corr. media = 0,30.")
    diversificacion_interactiva(50, 0.30)

## 8. Un ejemplo con datos

El archivo `retornos_ejemplo.csv` tiene 120 rendimientos mensuales de cinco carteras sectoriales y un índice de mercado. Están **simulados a partir de un modelo de un factor conocido** (ver `data/README.md` en el repositorio), justamente para poder comparar lo que estimamos con la verdad.

In [ ]:
datos = pd.read_csv("retornos_ejemplo.csv", index_col="fecha")
sectores = [c for c in datos.columns if c != "Mercado"]

resumen = pd.DataFrame({
    "media anual (%)": datos.mean() * 12 * 100,
    "desvío anual (%)": datos.std(ddof=1) * np.sqrt(12) * 100,
})
resumen["Sharpe (rf=3%)"] = (resumen["media anual (%)"] / 100 - 0.03) / (resumen["desvío anual (%)"] / 100)
display(resumen.round(2))
print("Períodos: %d meses (%s a %s)" % (len(datos), datos.index[0], datos.index[-1]))

In [ ]:
# Betas estimadas por regresión y rendimiento exigido por el CAPM.
var_M_emp = datos["Mercado"].var(ddof=1)
mu_M_emp = datos["Mercado"].mean() * 12
rf_emp = 0.03
beta_verdadera = {"Energia": 1.35, "Banca": 1.20, "Consumo": 0.80,
                  "Tecnologia": 1.55, "Utilities": 0.55}

filas = []
for s in sectores:
    b = datos[s].cov(datos["Mercado"]) / var_M_emp
    sd_s = datos[s].std(ddof=1) * np.sqrt(12)
    sd_M_emp = datos["Mercado"].std(ddof=1) * np.sqrt(12)
    _, _, _, r2_s = eri.descomposicion_riesgo(sd_s, b, sd_M_emp)
    filas.append({"activo": s,
                  "beta estimada": b,
                  "beta verdadera": beta_verdadera[s],
                  "error": b - beta_verdadera[s],
                  "R2": r2_s,
                  "CAPM exige (%)": eri.capm(b, rf_emp, mu_M_emp) * 100,
                  "realizado (%)": datos[s].mean() * 12 * 100})
capm_emp = pd.DataFrame(filas).set_index("activo")
display(capm_emp.round(3))

Dos lecturas, y la segunda es la importante:

1. **El ordenamiento se recupera bien.** Tecnología es el activo más sensible al ciclo y Utilities el menos; las betas estimadas respetan ese orden y el $R^2$ ordena igual.
2. **Las betas se estiman con mucho ruido.** Energía tiene beta verdadera 1,35 y estimada 1,06: un error del 21 %. Y eso con 120 meses —diez años de datos— y un modelo que *sabemos* que es exactamente el correcto, sin cambios de régimen, sin no linealidades, sin errores de medición.

> ⚠️ **Cuidado.** Ésta es la advertencia práctica de la clase: el CAPM es una teoría de equilibrio sobre valores esperados, pero se aplica con estimaciones. Una beta estimada no es la beta. Con datos reales —donde el modelo verdadero no es el de un factor, los parámetros cambian en el tiempo y la muestra es corta— el problema es considerablemente peor. Ninguna de las decisiones de inversión que se toman a partir de una beta puntual soporta el peso que se le suele poner encima.

In [ ]:
# Frontera eficiente estimada con los datos.
mu_emp = (datos[sectores].mean() * 12).values
Sigma_emp = (datos[sectores].cov() * 12).values

w_gmv_e, mu_gmv_e, sd_gmv_e = eri.cartera_gmv(mu_emp, Sigma_emp)
w_tan_e, mu_tan_e, sd_tan_e = eri.cartera_tangente(mu_emp, Sigma_emp, rf_emp)
sharpe_e = eri.ratio_sharpe(mu_tan_e, sd_tan_e, rf_emp)

rango_e = np.linspace(0.0, 0.30, 300)
sd_front_e = eri.frontera_sigma(rango_e, mu_emp, Sigma_emp)

fig, ax = eri.figura("Frontera estimada con los datos del CSV", "$\\sigma_p$", "$\\mu_p$")
efi_e = rango_e >= mu_gmv_e
ax.plot(sd_front_e[efi_e], rango_e[efi_e], color=eri.NAVY, label="frontera eficiente")
ax.plot(sd_front_e[~efi_e], rango_e[~efi_e], color=eri.GRIS, linestyle="--")
sd_cml_e = np.linspace(0, 0.45, 50)
ax.plot(sd_cml_e, rf_emp + sharpe_e * sd_cml_e, color=eri.ORO, label="CML")
ax.scatter(np.sqrt(np.diag(Sigma_emp)), mu_emp, color="black", s=28, zorder=5)
for nombre, s_, m_ in zip(sectores, np.sqrt(np.diag(Sigma_emp)), mu_emp):
    ax.annotate(nombre, (s_, m_), textcoords="offset points", xytext=(6, -3), fontsize=8)
ax.scatter([sd_tan_e], [mu_tan_e], color=eri.ORO, s=70, zorder=6, edgecolor="white")
ax.set_xlim(0, 0.45); ax.set_ylim(0, 0.30)
ax.legend(loc="lower right")
plt.show()

display(eri.tabla({"GMV": w_gmv_e, "Tangente": w_tan_e}, indice=sectores))
eri.resaltar("Ratio de Sharpe del tangente estimado", sharpe_e)

El tangente estimado carga fortísimo en Banca y vende en corto Consumo. **No hay que creerle.** Es el problema clásico de la optimización media-varianza: la cartera tangente es extremadamente sensible a los rendimientos esperados estimados, que son justamente lo que peor se estima. Banca tuvo la mejor media realizada de la muestra por azar, y el optimizador lo interpreta como una oportunidad. Es la razón por la cual, en la práctica, se usan estimadores contraídos (Black-Litterman, shrinkage de Ledoit-Wolf) o directamente se opera cerca de la GMV, que solo depende de $\Sigma$ —bastante mejor estimada que $\mu$.

<a name="sec9"></a>
## 9. Ejercicios resueltos

Los nueve ejercicios de la guía de la Clase 4. Cada uno cierra con un `assert` que verifica el resultado contra la solución analítica.

### Ejercicio 1 — Rendimiento y varianza de una cartera de dos activos

Dos activos con $\mu_A = 10\%$, $\mu_B = 18\%$; desvíos $\sigma_A = 15\%$, $\sigma_B = 30\%$; y correlación $\rho_{AB} = 0{,}2$. Se arma una cartera con $w_A = 0{,}6$ y $w_B = 0{,}4$. Calcule el rendimiento esperado y el desvío, y compare este último con el promedio ponderado de los desvíos individuales.

In [ ]:
mu_e1 = np.array([0.10, 0.18])
sd_e1 = np.array([0.15, 0.30])
rho_e1 = 0.20
Sigma_e1 = np.array([[sd_e1[0]**2,                    rho_e1*sd_e1[0]*sd_e1[1]],
                     [rho_e1*sd_e1[0]*sd_e1[1],       sd_e1[1]**2            ]])
w_e1 = np.array([0.6, 0.4])

mu_p1, sd_p1 = eri.momentos_cartera(w_e1, mu_e1, Sigma_e1)
promedio_sd = float(w_e1 @ sd_e1)

eri.resaltar("Covarianza", rho_e1 * sd_e1[0] * sd_e1[1])
eri.resaltar("Rendimiento esperado", mu_p1)
eri.resaltar("Varianza de la cartera", sd_p1 ** 2)
eri.resaltar("Desvío de la cartera", sd_p1)
eri.resaltar("Promedio ponderado de los desvíos", promedio_sd)
eri.resaltar("Ganancia por diversificar", promedio_sd - sd_p1)

assert abs(mu_p1 - 0.132) < 1e-9, "El rendimiento debería ser 13,2 %"
assert abs(sd_p1 - 0.163768) < 1e-5, "El desvío debería ser 16,38 %"

La media es exactamente el promedio ponderado; el desvío **no**: la diversificación lo reduce de 21 % a 16,38 %. Esa brecha de 4,6 puntos es el aporte de una correlación menor que 1, y se obtiene sin resignar rendimiento esperado.

### Ejercicio 2 — Cartera de mínima varianza global (dos activos)

Con los mismos datos, halle las ponderaciones de la GMV, su rendimiento esperado y su desvío.

Para dos activos, minimizar la varianza sujeta a $w_A + w_B = 1$ da

$$w_A^\star = \frac{\sigma_B^2 - \sigma_{AB}}{\sigma_A^2 + \sigma_B^2 - 2\sigma_{AB}}$$

In [ ]:
cov_e1 = rho_e1 * sd_e1[0] * sd_e1[1]
w_A_formula = (sd_e1[1]**2 - cov_e1) / (sd_e1[0]**2 + sd_e1[1]**2 - 2*cov_e1)
w_gmv1, mu_gmv1, sd_gmv1 = eri.cartera_gmv(mu_e1, Sigma_e1)

eri.resaltar("w_A por la fórmula de dos activos", w_A_formula)
eri.resaltar("w_A por la fórmula matricial", w_gmv1[0])
eri.resaltar("w_B", w_gmv1[1])
eri.resaltar("Rendimiento de la GMV", mu_gmv1)
eri.resaltar("Desvío de la GMV", sd_gmv1)
eri.resaltar("Desvío del activo menos riesgoso", sd_e1[0])

assert abs(w_A_formula - w_gmv1[0]) < 1e-12, "Las dos fórmulas deben coincidir"
assert abs(w_gmv1[0] - 0.857143) < 1e-5, "w_A debería ser 0,857"
assert abs(sd_gmv1 - 0.143427) < 1e-5, "El desvío de la GMV debería ser 14,34 %"
assert sd_gmv1 < sd_e1.min(), "La GMV debe tener menos riesgo que cualquier activo solo"

El resultado a subrayar: $\sigma_{\rm GMV} = 14{,}34\%$ es **menor que el desvío del activo menos riesgoso** (15 %). Combinando dos activos se logra menos riesgo que con cualquiera de ellos por separado. Es el argumento más limpio a favor de diversificar, y el que la regla del valor presente esperado no puede producir.

### Ejercicio 3 — Cartera sin riesgo con correlación perfecta negativa

Dos activos con $\sigma_1 = 12\%$, $\sigma_2 = 20\%$ y $\rho_{12} = -1$, con $\mu_1 = 8\%$ y $\mu_2 = 14\%$. Muestre que existe una cartera de riesgo nulo, halle sus ponderaciones y su rendimiento. ¿Qué relación de no arbitraje impone ese rendimiento?

Con $\rho_{12} = -1$ la varianza es un cuadrado perfecto: $\mathrm{Var}(r_p) = (w_1\sigma_1 - w_2\sigma_2)^2$, que se anula en $w_1 = \sigma_2/(\sigma_1+\sigma_2)$.

In [ ]:
sd1_e3, sd2_e3 = 0.12, 0.20
mu1_e3, mu2_e3 = 0.08, 0.14

w1_e3 = sd2_e3 / (sd1_e3 + sd2_e3)
w2_e3 = 1 - w1_e3
sd_e3 = abs(w1_e3 * sd1_e3 - w2_e3 * sd2_e3)
mu_e3 = w1_e3 * mu1_e3 + w2_e3 * mu2_e3

eri.resaltar("w_1", w1_e3)
eri.resaltar("w_2", w2_e3)
eri.resaltar("Desvío de la cartera", sd_e3)
eri.resaltar("Rendimiento de la cartera sin riesgo", mu_e3)

assert abs(w1_e3 - 0.625) < 1e-12, "w_1 debería ser 0,625"
assert sd_e3 < 1e-12, "La cartera debe tener riesgo exactamente nulo"
assert abs(mu_e3 - 0.1025) < 1e-12, "El rendimiento debería ser 10,25 %"

Por **no arbitraje**, ese 10,25 % debe coincidir con la tasa libre de riesgo $r_f$. Si no coincidiera, se podría construir una posición sin riesgo con ganancia segura: comprar la cartera y financiarla al activo seguro (si $r_f < 10{,}25\%$), o al revés. La correlación $-1$ replica exactamente la cobertura perfecta de los mercados de futuros del Ejercicio 8.

### Ejercicio 4 — Frontera con $N$ activos: mínima varianza global

Tres activos con $\boldsymbol{\mu} = (6\%, 10\%, 14\%)$ y rendimientos **no correlacionados**, con varianzas $0{,}04$, $0{,}09$ y $0{,}16$ (desvíos de 20 %, 30 % y 40 %). Calcule la GMV usando $\mathbf{w}_{\rm GMV} = \boldsymbol{\Sigma}^{-1}\mathbf{1}/C$.

In [ ]:
mu_e4 = np.array([0.06, 0.10, 0.14])
Sigma_e4 = np.diag([0.04, 0.09, 0.16])

A4, B4, C4, D4 = eri.frontera_abcd(mu_e4, Sigma_e4)
w_e4, mu_e4_p, sd_e4_p = eri.cartera_gmv(mu_e4, Sigma_e4)

print("Sigma inversa (diagonal):", np.round(np.diag(np.linalg.inv(Sigma_e4)), 4))
eri.resaltar("A", A4); eri.resaltar("C", C4)
display(eri.tabla({"ponderación": w_e4}, indice=["Activo 1", "Activo 2", "Activo 3"]))
eri.resaltar("Rendimiento de la GMV", mu_e4_p)
eri.resaltar("Desvío de la GMV", sd_e4_p)
eri.resaltar("Desvío del activo menos riesgoso", 0.20)

assert abs(w_e4[0] - 0.590164) < 1e-5, "w_1 debería ser 0,590"
assert abs(mu_e4_p - 0.0822951) < 1e-6, "El rendimiento debería ser 8,23 %"
assert abs(sd_e4_p - 0.153644) < 1e-5, "El desvío debería ser 15,37 %"

Con activos independientes, la GMV pondera cada uno de forma **inversamente proporcional a su varianza**: el menos riesgoso recibe el 59 % y el más riesgoso apenas el 15 %. Y el desvío resultante, 15,37 %, es notablemente menor que el 20 % del mejor activo individual: **con activos independientes la diversificación es especialmente potente**, tal como anticipaba el análisis del piso $\bar c = 0$ de la sección 1.

### Ejercicio 5 — Portafolio tangente y CML

Con los mismos tres activos y $r_f = 3\%$, halle el portafolio tangente, su rendimiento y desvío, el ratio de Sharpe del mercado y la ecuación de la CML.

In [ ]:
rf_e5 = 0.03
z = np.linalg.inv(Sigma_e4) @ (mu_e4 - rf_e5)
w_e5, mu_e5, sd_e5 = eri.cartera_tangente(mu_e4, Sigma_e4, rf_e5)
sharpe_e5 = eri.ratio_sharpe(mu_e5, sd_e5, rf_e5)

print("Premios por riesgo:", np.round(mu_e4 - rf_e5, 4))
print("Sigma^-1 (mu - rf*1):", np.round(z, 4), " suma =", round(z.sum(), 4))
display(eri.tabla({"ponderación": w_e5}, indice=["Activo 1", "Activo 2", "Activo 3"]))
eri.resaltar("Rendimiento del tangente", mu_e5)
eri.resaltar("Desvío del tangente", sd_e5)
eri.resaltar("Ratio de Sharpe", sharpe_e5)
print("\n  CML:  mu_p = %.4f + %.4f * sigma_p" % (rf_e5, sharpe_e5))

assert abs(w_e5[0] - 0.338558) < 1e-5, "w_1 debería ser 0,339"
assert abs(mu_e5 - 0.098871) < 1e-5, "El rendimiento debería ser 9,89 %"
assert abs(sharpe_e5 - 0.390601) < 1e-5, "El Sharpe debería ser 0,391"
assert sharpe_e5 > eri.ratio_sharpe(mu_e4_p, sd_e4_p, rf_e5), \
    "El tangente debe tener mejor Sharpe que la GMV"

Por el **teorema de separación de Tobin**, todo inversor combina este mismo $\mathbf{w}_T$ con el activo seguro; solo cambia la proporción según su aversión al riesgo. Un inversor muy averso pone 20 % en el tangente y 80 % en el activo seguro; uno poco averso puede poner 150 % en el tangente financiándose al $r_f$. **Ambos tienen la misma cartera riesgosa.**

El último `assert` verifica algo que la teoría exige: el tangente, por construcción, maximiza el ratio de Sharpe, de modo que debe superar a la GMV.

### Ejercicio 6 — CAPM: beta, rendimiento de equilibrio y valuación

El mercado tiene $\mu_M = 10\%$ y $\sigma_M = 18\%$, y $r_f = 3\%$. Un activo $i$ tiene $\mathrm{Cov}(r_i, r_M) = 0{,}0243$. Calcule su beta y su rendimiento de equilibrio. Si un analista proyecta un 9 %, ¿está infra o sobrevalorado?

In [ ]:
cov_iM, sd_M_e6, rf_e6, mu_M_e6 = 0.0243, 0.18, 0.03, 0.10

beta_e6 = eri.beta_capm(cov_iM, sd_M_e6 ** 2)
exigido = eri.capm(beta_e6, rf_e6, mu_M_e6)
proyectado = 0.09
alfa = proyectado - exigido

eri.resaltar("Varianza del mercado", sd_M_e6 ** 2)
eri.resaltar("Beta del activo", beta_e6)
eri.resaltar("Rendimiento exigido por la SML", exigido)
eri.resaltar("Rendimiento proyectado por el analista", proyectado)
eri.resaltar("Alfa", alfa)
print("\n  >> El activo está %s" % ("INFRAVALORADO (conviene comprar)" if alfa > 0
                                     else "SOBREVALORADO"))

assert abs(beta_e6 - 0.75) < 1e-12, "La beta debería ser 0,75"
assert abs(exigido - 0.0825) < 1e-12, "El exigido debería ser 8,25 %"
assert alfa > 0, "El activo debería estar infravalorado"

Como el rendimiento proyectado (9 %) supera al exigido por su riesgo sistemático (8,25 %), el activo ofrece un **alfa positivo** de 0,75 puntos y se ubica por encima de la SML. Al comprarlo, su precio sube y su rendimiento esperado cae hasta volver a la recta: **el alfa es, por definición, transitorio en equilibrio**.

### Ejercicio 7 — Descomposición del riesgo

Un activo tiene $\beta = 1{,}2$ y $\sigma_i = 28\%$; el mercado tiene $\sigma_M = 18\%$. Descomponga la varianza total y calcule la fracción sistemática.

In [ ]:
var_t7, var_s7, var_e7, r2_7 = eri.descomposicion_riesgo(0.28, 1.2, 0.18)

eri.resaltar("Varianza total", var_t7)
eri.resaltar("Varianza sistemática", var_s7)
eri.resaltar("Varianza no sistemática", var_e7)
eri.resaltar("Desvío residual", np.sqrt(var_e7))
eri.resaltar("Fracción sistemática (R2)", r2_7)
eri.resaltar("Fracción diversificable", 1 - r2_7)

assert abs(var_s7 - 0.046656) < 1e-9, "La varianza sistemática debería ser 0,046656"
assert abs(var_e7 - 0.031744) < 1e-9, "La varianza residual debería ser 0,031744"
assert abs(r2_7 - 0.595102) < 1e-5, "El R2 debería ser 59,5 %"
assert abs((var_s7 + var_e7) - var_t7) < 1e-12, "Las dos partes deben sumar el total"

El 59,5 % del riesgo del activo es sistemático y el 40,5 % restante es idiosincrático. Ese 40,5 % **se elimina por diversificación** y, por lo tanto, el mercado no lo remunera: solo el $\beta$ determina el premio por riesgo.

Un inversor que tenga este activo aislado en su cartera soporta el 100 % del riesgo pero cobra únicamente por el 59,5 %. **No diversificar tiene un costo, y no es que uno gane más por arriesgar más.**

### Ejercicio 8 — Cobertura de varianza mínima con futuros

Una empresa debe vender dentro de un mes 10.000 barriles de petróleo y quiere cubrirse con futuros (cada contrato es de 1.000 barriles). El desvío del cambio de precio spot es $\sigma_{\Delta S} = 3\%$, el del futuro $\sigma_{\Delta F} = 2{,}8\%$, y su correlación $\rho_{SF} = 0{,}9$. Halle el ratio de cobertura, el número de contratos y la reducción de riesgo.

In [ ]:
sdS_e8, sdF_e8, rho_e8 = 0.03, 0.028, 0.90
Q_spot, Q_contrato = 10_000, 1_000

h_e8 = eri.hedge_ratio(rho_e8, sdS_e8, sdF_e8)
N_contratos = h_e8 * Q_spot / Q_contrato
var_min = sdS_e8 ** 2 * (1 - rho_e8 ** 2)

eri.resaltar("Ratio de cobertura h*", h_e8)
eri.resaltar("Número óptimo de contratos", N_contratos)
eri.resaltar("Contratos a tomar (redondeado)", int(round(N_contratos)))
eri.resaltar("Varianza mínima del resultado", var_min)
eri.resaltar("Desvío con cobertura", np.sqrt(var_min))
eri.resaltar("Desvío sin cobertura", sdS_e8)
eri.resaltar("Reducción de la varianza", rho_e8 ** 2)
print("\n  >> La empresa está LARGA en el físico, así que toma posición CORTA en futuros.")

assert abs(h_e8 - 0.964286) < 1e-5, "h* debería ser 0,964"
assert abs(np.sqrt(var_min) - 0.013077) < 1e-5, "El desvío cubierto debería ser 1,31 %"
assert abs((1 - var_min / sdS_e8**2) - rho_e8**2) < 1e-12, \
    "La reducción de varianza debe ser exactamente rho^2"

La cobertura reduce la varianza en una proporción $\rho_{SF}^2 = 81\%$, llevando el desvío del 3 % al 1,31 %. El 19 % restante es **riesgo de base**: el futuro y el spot no se mueven exactamente juntos, y esa discrepancia no se puede cubrir con este instrumento.

Nótese que la reducción de varianza depende **solo de la correlación**, no de las volatilidades. Con $\rho_{SF} = 1$ la cobertura sería perfecta.

### Ejercicio 9 — Del factor de descuento estocástico al CAPM

Para un rendimiento bruto $R_i = 1 + r_i$, el precio de una unidad de rendimiento satisface $E[m R_i] = 1$, con $m$ el factor de descuento estocástico.

**(a)** Muestre que $E[R_i] - R_f = -R_f\,\mathrm{Cov}(m, R_i)$.
**(b)** Si $m = a - b\,R_M$, deduzca el CAPM.
**(c)** Verifique numéricamente con $R_f = 1{,}03$, $E[R_M] = 1{,}10$, $\mathrm{Var}(R_M) = 0{,}0324$ y $\mathrm{Cov}(R_M, R_i) = 0{,}0243$.

**(a)** Aplicando la condición al activo seguro, $E[m]R_f = 1$, de modo que $E[m] = 1/R_f$. Para un activo riesgoso, usando $E[mR_i] = E[m]E[R_i] + \mathrm{Cov}(m,R_i)$:

$$1 = \frac{E[R_i]}{R_f} + \mathrm{Cov}(m, R_i) \quad\Longrightarrow\quad E[R_i] - R_f = -R_f\,\mathrm{Cov}(m, R_i)$$

**(b)** Si $m = a - b R_M$, entonces $\mathrm{Cov}(m, R_i) = -b\,\mathrm{Cov}(R_M, R_i)$ y $E[R_i] - R_f = R_f\,b\,\mathrm{Cov}(R_M, R_i)$. Aplicando la igualdad al propio mercado se despeja $R_f b = (E[R_M]-R_f)/\mathrm{Var}(R_M)$, y sustituyendo se obtiene $E[R_i] - R_f = \beta_i(E[R_M]-R_f)$.

In [ ]:
Rf, ER_M, var_M9, cov_M9 = 1.03, 1.10, 0.0324, 0.0243

# El coeficiente b del SDF lineal, despejado de aplicar la relación al mercado.
b = (ER_M - Rf) / (Rf * var_M9)
a = 1 / Rf + b * ER_M                      # de E[m] = a - b*E[R_M] = 1/Rf

eri.resaltar("E[m] = 1/Rf", 1 / Rf)
eri.resaltar("Coeficiente b del SDF", b)
eri.resaltar("Coeficiente a del SDF", a)

# Vía SDF
cov_m_Ri = -b * cov_M9
prima_sdf = -Rf * cov_m_Ri
# Vía beta
beta9 = eri.beta_capm(cov_M9, var_M9)
prima_capm = beta9 * (ER_M - Rf)

eri.resaltar("Cov(m, R_i)", cov_m_Ri)
eri.resaltar("Prima por riesgo vía SDF", prima_sdf)
eri.resaltar("Beta del activo", beta9)
eri.resaltar("Prima por riesgo vía CAPM", prima_capm)
eri.resaltar("E[R_i] resultante", Rf + prima_capm)
eri.resaltar("En términos de rendimiento neto", Rf + prima_capm - 1)

assert abs(prima_sdf - prima_capm) < 1e-12, "Ambas vías deben dar la misma prima"
assert abs(beta9 - 0.75) < 1e-12, "La beta debería ser 0,75"
assert abs((Rf + prima_capm - 1) - 0.0825) < 1e-12, "Debería dar 8,25 %, igual que el Ej. 6"

El resultado es **idéntico al del Ejercicio 6**: 8,25 %. Los enfoques de Arrow-Borch (precios de estado) y de Markowitz-Sharpe (media-varianza) no son dos teorías, sino **dos lecturas del mismo problema de reparto del riesgo**.

La covarianza con el factor de descuento es negativa ($-0{,}0510$): el activo paga mal cuando $m$ es alto, es decir, cuando el consumo es escaso y la utilidad marginal alta. Por eso hay que pagar menos por él y por eso rinde más que el activo seguro. Es exactamente la lógica de la Unidad III.

## Síntesis

1. **La media de una cartera es un promedio ponderado; el riesgo no.** La covarianza gobierna el riesgo de una cartera grande, y por eso diversificar funciona. Pero hay un piso: el riesgo de covarianza no se elimina.
2. **La frontera eficiente** es la hipérbola $\sigma_p^2 = (C\mu_p^2 - 2A\mu_p + B)/D$, cuyo vértice es la GMV. Cuanto menor la correlación, más se comba hacia la izquierda.
3. **Separación de Tobin.** Con un activo libre de riesgo, todos eligen el mismo portafolio riesgoso tangente; la aversión al riesgo determina solo cuánto poner en él. La decisión se dicotomiza en «qué» y «cuánto».
4. **CAPM.** En equilibrio el tangente es el portafolio de mercado y $\mu_i = r_f + \beta_i(\mu_M - r_f)$. Solo se remunera el riesgo **sistemático**: el idiosincrático es evitable y por lo tanto no se paga.
5. **Los futuros** son un segundo canal de transferencia de riesgo, con $h^\star = \rho_{SF}\sigma_S/\sigma_F$ y una reducción de varianza igual a $\rho_{SF}^2$.
6. **Todo es lo mismo.** $\mu_i - r_f = -r_f\mathrm{Cov}(m, r_i)$: el CAPM es el caso en que el factor de descuento estocástico de la Unidad III es lineal en el mercado.
7. **Advertencia práctica.** La teoría es sobre valores esperados; la aplicación usa estimaciones ruidosas. Una beta estimada no es la beta, y una cartera tangente estimada suele ser un artefacto del error de estimación de $\boldsymbol{\mu}$.

## Para seguir

En la **Unidad V** llevamos la incertidumbre al lado de la **producción**: la teoría de la firma competitiva bajo incertidumbre de precios de Sandmo (1971). Veremos cómo una empresa neutral al riesgo, una aversa y una con función tipo Markowitz eligen su nivel de producción cuando el precio de venta es una variable aleatoria, y cómo la aversión al riesgo modifica la conocida regla «precio igual a costo marginal».

La conexión con esta clase es directa: la firma tipo Markowitz maximiza $E[\pi] - \tfrac{\rho}{2}\mathrm{Var}(\pi)$, exactamente el mismo criterio media-varianza que usamos acá, y con la misma justificación CARA-Normal de la Unidad II.

## Referencias

- Markowitz, H. (1952). Portfolio selection. *The Journal of Finance*, 7(1), 77–91.
- Sharpe, W. F. (1964). Capital asset prices: A theory of market equilibrium under conditions of risk. *The Journal of Finance*, 19(3), 425–442.
- Tobin, J. (1958). Liquidity preference as behavior towards risk. *The Review of Economic Studies*, 25(2), 65–86.
- Lintner, J. (1965). The valuation of risk assets and the selection of risky investments in stock portfolios and capital budgets. *The Review of Economics and Statistics*, 47(1), 13–37.
- Mossin, J. (1966). Equilibrium in a capital asset market. *Econometrica*, 34(4), 768–783.
- Merton, R. C. (1972). An analytic derivation of the efficient portfolio frontier. *Journal of Financial and Quantitative Analysis*, 7(4), 1851–1872.
- Borch, K. H. (1962). Equilibrium in a reinsurance market. *Econometrica*, 30(3), 424–444.
- Gravelle, H., & Rees, R. (2006). *Microeconomía* (3.ª ed.). Pearson.
- Varian, H. R. (1992). *Análisis microeconómico* (3.ª ed.). Antoni Bosch.
- Hull, J. C. (2018). *Options, Futures, and Other Derivatives* (10.ª ed.). Pearson.
- Cochrane, J. H. (2005). *Asset Pricing* (ed. rev.). Princeton University Press.

---

<div align="center">
<sub>Economía del Riesgo y de la Información (1.4.010) — UADE FCE — 2.º cuatrimestre 2026</sub><br>
<sub><a href="https://github.com/santiagoriverti/UADE_ERI">github.com/santiagoriverti/UADE_ERI</a></sub>
</div>